In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [2]:
rng = np.random.default_rng(100)

def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

In [3]:
GRID = list(range(5, 66, 10)) # Prices to test
size = 500 #Average number of customers per price point

sales_p = {p: sales(p, mean_customers=size, rng=rng) * p for p in GRID}  # Compute revenue for each customer

print("prices tested:", list(sales_p))
print("customers per price:", {p: len(v) for p, v in sales_p.items()})
print("total customers:", sum(len(v) for v in sales_p.values()))

prices tested: [5, 15, 25, 35, 45, 55, 65]
customers per price: {5: 525, 15: 474, 25: 485, 35: 495, 45: 512, 55: 524, 65: 505}
total customers: 3520


In [4]:
OPTIMAL_PROFIT = 11.925

cost = {p: float(len(v) * OPTIMAL_PROFIT - v.sum()) for p, v in sales_p.items()}

total_customers = sum(len(v) for v in sales_p.values())
total_revenue = sum(float(v.sum()) for v in sales_p.values())
total_cost = sum(cost.values())

print("cost per price:", {p: round(c, 2) for p, c in cost.items()})
print(f"\ncustomers tested:              {total_customers:,}")
print(f"revenue collected:             ${total_revenue:,.2f}")
print(f"revenue at the optimal price:  ${total_customers * OPTIMAL_PROFIT:,.2f}")
print(f"total cost of the grid search: ${total_cost:,.2f}")


cost per price: {5: 3635.62, 15: 1647.45, 25: 783.62, 35: 3277.88, 45: 3000.6, 55: 3168.7, 65: 5957.12}

customers tested:              3,520
revenue collected:             $20,505.00
revenue at the optimal price:  $41,976.00
total cost of the grid search: $21,471.00


In [5]:
Z = 1.96
revenue = {p: sales_p[p].mean() for p in GRID}  # Mean revenue per arriving customer

pairs = []
for i in range(len(GRID)):
    for j in range(i + 1, len(GRID)):
        a, b = GRID[i], GRID[j]
        delta_R = revenue[b] - revenue[a]  # Difference in mean revenue per arriving customer
        std_error = np.sqrt(np.var(sales_p[a], ddof=1) / len(sales_p[a])
                            + np.var(sales_p[b], ddof=1) / len(sales_p[b]))  # Standard Error
        ci_low, ci_high = delta_R - Z * std_error, delta_R + Z * std_error
        pairs.append([a, b, delta_R, std_error, ci_low, ci_high, ci_low > 0 or ci_high < 0])

compare = pd.DataFrame(pairs, columns=["price_a", "price_b", "delta", "std_error",
                                       "ci_low", "ci_high", "significant"])

flags = pd.DataFrame(".", index=GRID, columns=GRID)
for a, b, _, _, lo, hi, sig in pairs:
    if not sig:
        continue
    winner, loser = (b, a) if lo > 0 else (a, b)
    flags.loc[winner, loser] = ">"
    flags.loc[loser, winner] = "<"

dominated = [p for p in GRID if (flags.loc[p] == "<").any()]

print(flags.to_string())
print("\ndominated:", dominated)
print("survivors:", [p for p in GRID if p not in dominated])


   5  15 25 35 45 55 65
5   .  <  <  .  .  .  >
15  >  .  <  >  >  >  >
25  >  >  .  >  >  >  >
35  .  <  <  .  .  .  >
45  .  <  <  .  .  .  >
55  .  <  <  .  .  .  >
65  <  <  <  <  <  <  .

dominated: [5, 15, 35, 45, 55, 65]
survivors: [25]


In [6]:
def estimate_gradient(price, *, step=1.0, mean_customers=300, rng=None):
    if step <= 0:
        raise ValueError("step must be positive")
    rng = np.random.default_rng(rng)
    price = float(np.clip(price, 0, 75))
    price_low = max(0, price - step)
    price_high = min(75, price + step)
    low = price_low * sales(price_low, mean_customers=mean_customers, rng=rng)
    high = price_high * sales(price_high, mean_customers=mean_customers, rng=rng)
    if min(len(low), len(high)) < 2:
        raise ValueError("Need at least two customers per arm; increase mean_customers.")

    difference = high.mean() - low.mean()
    se = np.sqrt(low.var(ddof=1) / len(low) + high.var(ddof=1) / len(high))
    gap = price_high - price_low
    return {
        "price": price,
        "gradient": difference / gap,
        "gradient_se": se / gap,
        "ci_low": (difference - 1.645 * se) / gap,
        "ci_high": (difference + 1.645 * se) / gap,
        "customers": len(low) + len(high),
        "revenue": float(low.sum() + high.sum()),   # added to track revenue to calculate costs
    }

def gradient_ascent(start_price, *, iterations=80, step=1.0,
                    learning_rate=2.0, mean_customers=300, seed=100):
    """Use a fixed experiment budget and a decreasing learning rate."""
    if iterations < 1:
        raise ValueError("iterations must be at least 1")
    search_rng = np.random.default_rng(seed)
    price = float(np.clip(start_price, 0, 75))
    records = []
    for iteration in range(iterations):
        estimate = estimate_gradient(price, step=step,
                                     mean_customers=mean_customers, rng=search_rng)
        rate = learning_rate / np.sqrt(iteration + 1)
        move = np.clip(rate * estimate["gradient"], -2.0, 2.0)
        next_price = float(np.clip(price + move, 0, 75))
        records.append({"iteration": iteration + 1, **estimate,
                        "learning_rate": rate, "next_price": next_price})
        price = next_price
    return pd.DataFrame(records)


In [ ]:
##Non of the gradient steps are significant 

path = gradient_ascent(25, iterations=20, step=1.0, learning_rate=2.0, mean_customers=250, seed=100)

print(path[["iteration", "price", "gradient", "ci_low", "ci_high", "next_price", "customers"]].round(3).to_string(index=False))

customers = int(path.customers.sum())
revenue = float(path.revenue.sum())

print(f"\nfinal price: ${path['next_price'].iloc[-1]:.2f}")
print(f"customers tested:                  {customers:,}")
print(f"revenue collected:                 ${revenue:,.2f}")
print(f"revenue at the optimal price:      ${customers * OPTIMAL_PROFIT:,.2f}")
print(f"total cost of the gradient search: ${customers * OPTIMAL_PROFIT - revenue:,.2f}")

 iteration  price  gradient  ci_low  ci_high  next_price  customers
         1 25.000    -0.044  -0.935    0.846      24.911        535
         2 24.911     0.879  -0.051    1.809      26.154        483
         3 26.154    -0.748  -1.699    0.203      25.291        492
         4 25.291     0.723  -0.190    1.637      26.014        511
         5 26.014     0.228  -0.759    1.215      26.218        453
         6 26.218     0.336  -0.623    1.294      26.492        501
         7 26.492     0.413  -0.603    1.429      26.804        463
         8 26.804     0.064  -0.883    1.010      26.849        541
         9 26.849     0.271  -0.700    1.242      27.030        520
        10 27.030     0.375  -0.627    1.377      27.267        481
        11 27.267    -0.556  -1.597    0.486      26.932        459
        12 26.932    -0.040  -1.020    0.940      26.909        495
        13 26.909     0.306  -0.706    1.318      27.079        477
        14 27.079    -0.138  -1.124    0.848    

In [8]:
grid_cost = sum(len(v) for v in sales_p.values()) * OPTIMAL_PROFIT - sum(float(v.sum()) for v in sales_p.values())
gradient_cost = int(path.customers.sum()) * OPTIMAL_PROFIT - float(path.revenue.sum())

print(f"cost of the grid search:     ${grid_cost:,.2f}")
print(f"cost of the gradient search: ${gradient_cost:,.2f}")
print(f"total cost of the search:    ${grid_cost + gradient_cost:,.2f}")

cost of the grid search:     $21,471.00
cost of the gradient search: $2,953.69
total cost of the search:    $24,424.69



2. Tracks the total number of consumers you have experimented with (at the optimum, each consumer has an expected profit of $11.925)

- The optimal price is between 0 and 70: There is no reason to consider prices outside that range
- There are multiple prices that appear optimal, but only one is globally optimal



- Provide a 1-3 page write-up explaining your design, including tables and plots as necessary:
    1. A one-paragraph summary of the problem
    2. Two paragraphs about how your design handles uncertainty about changes in revenue and selects prices
    3. Two paragraphs about challenges you faced and how you resolved them
    4. One paragraph that summarizes your results. In particular, how many consumers did you experiment on, what were your revenues during the experiments, and what percentage of the optimal profit ($11.925 per customer) did you capture?